# Structured R1 + Free-Form Rationale Consolidation

This notebook evaluates the fourth consolidation-output format used in the thesis:

**Structured R1 + free-form rationale + final binary label.**

It follows the previous consolidation experiments:

1. **Binary only**
2. **Structured R1**
3. **Free-form rationale**
4. **Structured R1 + rationale** ← this notebook

The purpose of this experiment is to test what happens when the model is required to produce **both** the categorical Structured R1 assessments **and** an additional natural-language rationale before returning the final `NORMAL` / `ANOMALOUS` decision.

## Relationship to Structured R1

The original Structured R1 experiment exposes fixed categorical assessments for the main evidence dimensions:

- `participation_assessment`
- `local_temporal_assessment`
- `global_temporal_assessment`
- `temporal_assessment`
- `semantic_assessment`
- `decisive_dimension`
- final binary `label`

Those fields make the model's observable evidence assessment auditable.

In this experiment, **all of those Structured R1 fields are retained unchanged**.

The only additional model-facing output is one free-form text field:

```text
chain_of_thought
```

Despite the historical field name used in the executed experiment, this value is treated in the thesis and repository as a **free-form rationale / observable explanation**, not as a guaranteed faithful representation of the model's hidden internal reasoning.

The resulting output structure is therefore conceptually:

```text
structured participation assessment
        ↓
structured local temporal assessment
        ↓
structured global temporal assessment
        ↓
structured combined temporal assessment
        ↓
structured semantic assessment
        ↓
structured decisive dimension
        ↓
free-form rationale
        ↓
final NORMAL / ANOMALOUS label
```

## Controlled comparison

Relative to the original Structured R1 experiment, the following are kept unchanged:

- the exact 400 development cases;
- the same participant-centric evidence packet;
- the same participation evidence;
- the same filtered turns;
- the same local temporal features;
- the same global temporal features;
- the same coarse semantic summaries;
- the same focused semantic summaries;
- the same frozen NORMAL temporal reference;
- the same independent-normality decision policy;
- the same Qwen2.5-Omni model;
- the same deterministic decoding configuration;
- the same Structured R1 categorical fields.

The experiment explicitly loads the saved Structured R1 source prompt and preserves all prompt text before the output section.

The **only model-facing change relative to Structured R1** is the addition of the free-form rationale field.

This makes the experiment a controlled test of whether appending natural-language explanation to the structured assessment scaffold changes the model's final classification behaviour.

## Thesis-reported result

The Structured R1 + rationale configuration produces:

| Format | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| **R1 + rationale** | **60/100** | **94/100** | **93/100** | **100/100** | **86.75%** |

The corresponding binary confusion matrix is:

```text
                Pred NORMAL   Pred ANOMALOUS
Gold NORMAL          60             40
Gold ANOMALOUS       13            287
```

The overall accuracy is very similar to Structured R1, but the class-specific behaviour changes substantially:

- NORMAL preservation decreases from 78/100 to 60/100;
- LAG detection increases from 80/100 to 94/100;
- Wrong Partner detection increases from 88/100 to 93/100;
- Silent Partner remains 100/100.

This shift is important because the underlying evidence is unchanged. Adding a free-form rationale after the structured assessments makes the reasoner substantially more anomaly-sensitive, particularly for temporal deviations.

Together with the Binary-only, Structured R1, and rationale-only experiments, this result shows that **output format is part of the effective consolidation mechanism rather than merely a passive reporting choice**.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, inspection cell, and evaluation result in this notebook is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. Then select:

`Runtime → Restart session`

and continue from the next cell.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 150.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 142.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 57.2 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and configure artifact paths

The folder name still contains `1_2_3sec` for historical reasons. The finalized 400-case database is loaded from the exact path below.

In [ ]:
# from google.colab import drive

# drive.mount("/content/drive")


# from pathlib import Path
# from collections import Counter
# import copy
# import json

# import pandas as pd
# from IPython.display import display


# OUT_DIR = Path(
#     "/content/drive/MyDrive/"
#     "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
# )


# REFERENCE_BASE_STATS_PATH = (
#     OUT_DIR
#     / "frozen_reference_base_statistics.json"
# )


# REFERENCE_SHIFT_STATS_PATH = (
#     OUT_DIR
#     / "frozen_reference_global_shift_statistics.json"
# )


# FINAL_DATABASE_PATH = (
#     OUT_DIR
#     / (
#         "consolidation_all_400_cases_with_"
#         "temporal_and_semantic_summaries.json"
#     )
# )


# MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

# MODEL_PATH = Path(
#     "/content/drive/MyDrive/Qwen2.5-Omni-7B"
# )


# required_paths = {
#     "Frozen base statistics": (
#         REFERENCE_BASE_STATS_PATH
#     ),

#     "Frozen global-shift statistics": (
#         REFERENCE_SHIFT_STATS_PATH
#     ),

#     "Final 400-case database": (
#         FINAL_DATABASE_PATH
#     ),

#     "Local Qwen checkpoint": (
#         MODEL_PATH
#     ),
# }


# print("=" * 88)
# print("ARTIFACT PATH AUDIT")
# print("=" * 88)

# for name, path in required_paths.items():

#     print(
#         f"{name}:",
#         path,
#     )

#     print(
#         "  exists:",
#         path.exists(),
#     )


# assert OUT_DIR.exists(), (
#     f"Project directory not found: {OUT_DIR}"
# )

# assert REFERENCE_BASE_STATS_PATH.exists(), (
#     "Frozen base-statistics file not found: "
#     f"{REFERENCE_BASE_STATS_PATH}"
# )

# assert REFERENCE_SHIFT_STATS_PATH.exists(), (
#     "Frozen global-shift-statistics file not found: "
#     f"{REFERENCE_SHIFT_STATS_PATH}"
# )

# assert FINAL_DATABASE_PATH.exists(), (
#     "Final consolidation database not found: "
#     f"{FINAL_DATABASE_PATH}"
# )

# assert MODEL_PATH.exists(), (
#     "Local Qwen checkpoint not found: "
#     f"{MODEL_PATH}"
# )



from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


# ============================================================
# MAIN PROJECT DIRECTORY
# ============================================================

OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


# ============================================================
# INPUT ARTIFACTS
# ============================================================

REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


# ============================================================
# MODEL
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"


MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


# ============================================================
# CHAIN-OF-THOUGHT EXPERIMENT CACHE
# ============================================================

CHAIN_OF_THOUGHT_EXPERIMENT_DIR = (
    OUT_DIR
    / (
        "reasoning_r1_full_semantics_"
        "normal_references_only_chain_of_thought"
    )
)


CHAIN_OF_THOUGHT_CACHE_PATH = (
    CHAIN_OF_THOUGHT_EXPERIMENT_DIR
    / "predictions_cache.json"
)


print("=" * 88)
print("CHAIN-OF-THOUGHT CACHE RESET")
print("=" * 88)

print(
    "Cache path:",
    CHAIN_OF_THOUGHT_CACHE_PATH,
)


if CHAIN_OF_THOUGHT_CACHE_PATH.exists():

    CHAIN_OF_THOUGHT_CACHE_PATH.unlink()

    print(
        "Deleted previous cache successfully."
    )

else:

    print(
        "No previous cache file was found."
    )


print(
    "Cache exists after reset:",
    CHAIN_OF_THOUGHT_CACHE_PATH.exists(),
)


# ============================================================
# REQUIRED PATH AUDIT
# ============================================================

required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print()
print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


# ============================================================
# SAFETY CHECKS
# ============================================================

assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)


assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)


assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)


assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)


assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
CHAIN-OF-THOUGHT CACHE RESET
Cache path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought/predictions_cache.json
Deleted previous cache successfully.
Cache exists after reset: False

ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the frozen NORMAL reference statistics

Both frozen JSON files contain separate reference profiles. This experiment deliberately extracts and retains only:

```text
reference_profile = NORMAL
```

The lag reference profiles are not included in the binary-only experiment state at this stage.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the final 400-case database

This cell verifies:

- exactly 400 unique cases;
- exactly 100 cases per family;
- 100 `NORMAL` and 300 `ANOMALOUS` binary labels;
- participant-level `speaks` agrees with the final filtered VAD turns;
- every sample has all four coarse and all four focused summaries;
- no focused summary contains the legacy `speaks` field.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-level `speaks` overview

The field is derived exclusively from each sample's final filtered VAD turns.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive case inspection

The inspector supports all four families:

- `NORMAL`
- `WRONG_PARTNER`
- `LAG`
- `SILENT_PARTNER`

It can display participant metadata, VAD-derived `speaks`, filtered turns, temporal features, semantic summaries, and the complete sample JSON.

The direct function can also be used without widgets:

```python
inspect_case(
    family="normal",
    sample_index=0,
)
```

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker


In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


# Ηelper Functions For Qwen Calls

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Frozen NORMAL reference text

This is the same reference-text construction used by the original binary Experiment 2 and the original R1 experiment. No anomaly-specific reference profile is added.


In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

# R1 full-semantics input and reasoning helpers

The model-facing input projection is copied from the original binary consolidation notebook and is identical to the original R1 experiment. It passes exactly the same participation information, filtered turns, local and global temporal features, coarse summaries, and focused summaries.


In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The model-facing input projection is copied from the original
# binary consolidation notebook:
#   - same participation fields
#   - same filtered turns
#   - same local temporal fields
#   - same global temporal fields
#   - same coarse summaries
#   - same focused summaries
#
# The R1 source prompt is loaded from its exact saved
# prompt_template.txt file.
#
# Relative to the original R1 experiment, the only model-facing
# change is one additional free-form chain_of_thought JSON field.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 1024
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "chain_of_thought",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


OLD_R1_STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

After assigning all fixed structured assessments, provide one additional
free-form chain_of_thought field.

After assigning all fixed structured assessments, provide one additional
free-form chain_of_thought field.

In chain_of_thought, provide a concise step-by-step rationale using at most
7 short numbered steps and no more than 200 words in total:

1. assess participation,
2. assess the local temporal evidence,
3. assess the global temporal evidence,
4. combine the local and global evidence using the existing temporal policy,
5. assess semantic compatibility,
6. explain which dimension is decisive,
7. explain how the final binary label follows from the existing policy.

Explicitly discuss conflicting, weak, or LIMITED evidence when present.
Explain how such evidence affects the decision, but do so concisely.

Do not repeat all numerical feature values.
Do not restate the prompt or the complete evidence packet.
Include only the evidence that directly supports the structured assessments
and the final binary decision.

The chain_of_thought must use only the evidence supplied in the prompt.
It must not introduce new evidence, thresholds, decision rules, labels,
anomaly subtypes, or delay magnitudes.

The chain_of_thought value must be one non-empty valid JSON string.
Do not use unescaped quotation marks or literal line breaks inside the
JSON string.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "chain_of_thought": "Free-form step-by-step reasoning as one valid JSON string",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key not in parsed:

                continue


            if key == "chain_of_thought":

                if isinstance(
                    parsed[
                        key
                    ],
                    str,
                ):

                    normalized[
                        key
                    ] = (
                        parsed[
                            key
                        ].strip()
                    )

                else:

                    schema_errors.append(
                        "chain_of_thought must be a JSON string."
                    )

            else:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for (
            key,
            allowed_values,
        ) in REASONING_ALLOWED_VALUES.items():

            if normalized[
                key
            ] not in allowed_values:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        if not normalized[
            "chain_of_thought"
        ]:

            schema_errors.append(
                "chain_of_thought is empty or missing."
            )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        '"chain_of_thought"'
        in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    assert (
        "Do not provide free-form reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "Relative to the original R1 experiment, the "
            "structured OUTPUT block is unchanged except "
            "for one added chain_of_thought JSON field."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "original R1 structured OUTPUT block -> "
            "same block plus chain_of_thought"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
                "chain_of_thought": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "chain_of_thought": (
                parsed_result[
                    "chain_of_thought"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "chain_of_thought": (
                None
                if record is None
                else record.get(
                    "chain_of_thought"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ]
            !=
            results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "chain_of_thought",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


# R1 — Full Semantics, NORMAL References Only, Structured Reasoning + Chain of Thought

**Exact baseline:** the original R1 experiment, whose model-facing inputs, decision policies, six structured assessments, and final binary label are retained.

The only added output is:

```text
"chain_of_thought": "free-form step-by-step reasoning"
```

No decision rule, threshold, reference, feature, model configuration, case, or evaluation criterion is changed.


In [ ]:

# ============================================================
# R1 CONFIGURATION
# FULL SEMANTICS + NORMAL REFERENCES ONLY + ORIGINAL R1 FIELDS + CHAIN OF THOUGHT
#
# Exact source:
#   Experiment 2 — Independent Normality Requirements
#
# Only change relative to the original R1:
#   add one chain_of_thought string field to the same output schema
# ============================================================

R1_CONFIG = prepare_reasoning_experiment(
    experiment_version=(
        "reasoning_r1_full_semantics_"
        "normal_references_only_chain_of_thought"
    ),

    source_experiment_name=(
        "binary_only_consolidation_"
        "normal_definition_v2"
    ),

    experiment_title=(
        "R1 — Full Semantics, No LAG Profiles, "
        "Structured Reasoning + Chain of Thought"
    ),

    required_source_markers=[
        "NORMAL requires three independent properties",
        "FROZEN NORMAL LOCAL-TIMING REFERENCE",
        "FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE",
        "Coarse semantic information:",
        "Focused semantic information:",
    ],

    forbidden_source_markers=[
        "FROZEN NON-NORMAL LOCAL-TIMING REFERENCE PROFILES",
        "Frozen LAG_2 local reference pattern",
        "Frozen LAG_3 local reference pattern",
        "TEMPORAL PROFILE COMPARISON",
        "ORDERED AND INDEPENDENT EVIDENCE ASSESSMENT",
    ],

    temporal_profiles_used=[
        "NORMAL",
    ],

    assessment_policy=(
        "Original Experiment 2 independent-normality "
        "requirements, unchanged."
    ),
)


R1 — Full Semantics, No LAG Profiles, Structured Reasoning + Chain of Thought — CONFIGURATION READY
Experiment version: reasoning_r1_full_semantics_normal_references_only_chain_of_thought
Source experiment: binary_only_consolidation_normal_definition_v2
Source prompt SHA256: ed2d68e14f1fa667650b11ef8e2c75a7f03c48c7e3faa1dfd0dabecaa14bbace
Reasoning prompt SHA256: 0c53fc8b4408d53969bc5ba163b0e8d7800f2d8cf983f69464c3430834c44298
Semantic input: coarse_and_focused
Focused summaries used: True
Temporal profiles: ['NORMAL']
Assessment policy: Original Experiment 2 independent-normality requirements, unchanged.
Only source-prompt change: original R1 structured OUTPUT block -> same block plus chain_of_thought
Example prompt characters: 26868
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought/predictions_cache.json
Existing cache: False


In [ ]:

# ============================================================
# RUN R1
# ============================================================

R1_CACHE = run_reasoning_experiment(
    R1_CONFIG
)


Created cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought/predictions_cache.json


reasoning_r1_full_semantics_normal_references_only_chain_of_thought:   0%|          | 0/400 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


# ============================================================
# LOAD FINAL SAVED PREDICTIONS
# THE CACHE WAS DELETED, BUT THE FINAL CSV STILL EXISTS
# ============================================================

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "reasoning_r1_full_semantics_normal_references_only_chain_of_thought"
)


PREDICTIONS_PATH = (
    EXPERIMENT_DIR
    / "predictions_all_400.csv"
)


assert PREDICTIONS_PATH.exists(), (
    "The final predictions file was not found:\n"
    f"{PREDICTIONS_PATH}"
)


cache_df = pd.read_csv(
    PREDICTIONS_PATH
)


print(
    "Loaded final samples:",
    len(cache_df)
)

print(
    "Predictions path:",
    PREDICTIONS_PATH
)


assert len(cache_df) == 400, (
    "Expected 400 final predictions, but loaded "
    f"{len(cache_df)}."
)


# ============================================================
# PARSE MODES
# ============================================================

print("\nPARSE MODES")

display(
    cache_df[
        "parse_mode"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "parse_mode"
    )
    .reset_index(
        name="count"
    )
)


# ============================================================
# OUTPUT COMPLETENESS CHECKS
# ============================================================

cache_df[
    "has_chain_of_thought"
] = (
    cache_df[
        "raw_output"
    ]
    .fillna("")
    .astype(str)
    .str.contains(
        '"chain_of_thought"',
        regex=False
    )
)


cache_df[
    "has_label"
] = (
    cache_df[
        "raw_output"
    ]
    .fillna("")
    .astype(str)
    .str.contains(
        '"label"',
        regex=False
    )
)


cache_df[
    "ends_with_closing_brace"
] = (
    cache_df[
        "raw_output"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.rstrip("`")
    .str.strip()
    .str.endswith("}")
)


print("\nOUTPUT COMPLETENESS")

display(
    cache_df[
        [
            "case_id",
            "case_family",
            "parse_mode",
            "has_chain_of_thought",
            "has_label",
            "ends_with_closing_brace",
        ]
    ]
)


# ============================================================
# SUMMARY
# ============================================================

print("\nCOMPLETENESS SUMMARY")

completeness_summary = pd.DataFrame({
    "criterion": [
        "Has chain_of_thought",
        "Has label",
        "Ends with closing brace",
    ],

    "count": [
        int(
            cache_df[
                "has_chain_of_thought"
            ].sum()
        ),

        int(
            cache_df[
                "has_label"
            ].sum()
        ),

        int(
            cache_df[
                "ends_with_closing_brace"
            ].sum()
        ),
    ],
})


completeness_summary[
    "percentage"
] = (
    100.0
    * completeness_summary[
        "count"
    ]
    / len(cache_df)
).round(2)


display(
    completeness_summary
)


# ============================================================
# RESTORE THE VARIABLE USED BY THE ANALYSIS CELLS
# ============================================================

cot_df = cache_df.copy()


R1_EVALUATION = {
    "results_df": cot_df
}


print(
    "\nRestored cot_df and R1_EVALUATION "
    f"with {len(cot_df)} cases."
)

Loaded final samples: 400
Predictions path: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought/predictions_all_400.csv

PARSE MODES


,parse_mode,count
0,structured_json,400



OUTPUT COMPLETENESS


,case_id,case_family,parse_mode,has_chain_of_thought,has_label,ends_with_closing_brace
0,consolidation_normal_000,normal,structured_json,True,True,False
1,consolidation_normal_001,normal,structured_json,True,True,False
2,consolidation_normal_002,normal,structured_json,True,True,False
3,consolidation_normal_003,normal,structured_json,True,True,False
4,consolidation_normal_004,normal,structured_json,True,True,False
...,...,...,...,...,...,...
395,consolidation_silent_partner_095,silent_partner,structured_json,True,True,False
396,consolidation_silent_partner_096,silent_partner,structured_json,True,True,False
397,consolidation_silent_partner_097,silent_partner,structured_json,True,True,False
398,consolidation_silent_partner_098,silent_partner,structured_json,True,True,False



COMPLETENESS SUMMARY


,criterion,count,percentage
0,Has chain_of_thought,400,100.0
1,Has label,400,100.0
2,Ends with closing brace,0,0.0



Restored cot_df and R1_EVALUATION with 400 cases.


In [ ]:

# # ============================================================
# # EVALUATE R1
# # ============================================================

# R1_EVALUATION = evaluate_reasoning_experiment(
#     R1_CONFIG
# )

# ============================================================
# RESTORE R1 EVALUATION FROM THE FINAL SAVED CSV
# DO NOT READ THE DELETED CACHE
# ============================================================

cot_df = cache_df.copy()

assert len(cot_df) == 400, (
    f"Expected 400 predictions, found {len(cot_df)}."
)


assert cot_df["case_id"].is_unique, (
    "Duplicate case IDs were found."
)


assert cot_df["prediction"].isin(
    [
        "NORMAL",
        "ANOMALOUS",
    ]
).all(), (
    "At least one final prediction is invalid."
)


R1_EVALUATION = {
    "results_df": cot_df
}


print(
    "Restored R1_EVALUATION from predictions_all_400.csv"
)

print(
    "Total cases:",
    len(R1_EVALUATION["results_df"])
)

print(
    "Valid predictions:",
    R1_EVALUATION[
        "results_df"
    ][
        "prediction"
    ].isin(
        [
            "NORMAL",
            "ANOMALOUS",
        ]
    ).sum()
)

Restored R1_EVALUATION from predictions_all_400.csv
Total cases: 400
Valid predictions: 400


In [ ]:
from pathlib import Path

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "reasoning_r1_full_semantics_normal_references_only_chain_of_thought"
)

files_to_check = [
    "predictions_cache.json",
    "predictions_all_400.csv",
    "reasoning_assessments.csv",
    "classification_errors.csv",
    "prompt_template.txt",
    "prompt_diff_vs_source.txt",
]

print("=" * 90)
print("EXPERIMENT FILE AUDIT")
print("=" * 90)

for filename in files_to_check:
    path = EXPERIMENT_DIR / filename

    print(
        f"{filename}:",
        "EXISTS" if path.exists() else "MISSING"
    )

    if path.exists():
        print(
            "  size:",
            path.stat().st_size,
            "bytes"
        )

EXPERIMENT FILE AUDIT
predictions_cache.json: MISSING
predictions_all_400.csv: EXISTS
  size: 2668178 bytes
reasoning_assessments.csv: EXISTS
  size: 414017 bytes
classification_errors.csv: EXISTS
  size: 353193 bytes
prompt_template.txt: EXISTS
  size: 20209 bytes
prompt_diff_vs_source.txt: EXISTS
  size: 3900 bytes


In [ ]:
import pandas as pd
from IPython.display import display

PREDICTIONS_PATH = (
    EXPERIMENT_DIR
    / "predictions_all_400.csv"
)

assert PREDICTIONS_PATH.exists(), (
    f"Predictions file not found: {PREDICTIONS_PATH}"
)

cot_df = pd.read_csv(
    PREDICTIONS_PATH
)

print("Loaded predictions:", cot_df.shape)
print("Columns:", cot_df.columns.tolist())

display(
    cot_df.head()
)

Loaded predictions: (400, 26)
Columns: ['case_id', 'source_group_id', 'case_family', 'case_variant', 'gold_label', 'prediction', 'participation_assessment', 'local_temporal_assessment', 'global_temporal_assessment', 'temporal_assessment', 'semantic_assessment', 'decisive_dimension', 'chain_of_thought', 'parse_mode', 'schema_exact', 'schema_errors', 'input_token_count', 'elapsed_seconds', 'raw_output', 'generation_error', 'valid_prediction', 'correct', 'normal_with_invalid_participation', 'normal_with_anomalous_temporal', 'normal_with_incompatible_semantics', 'reasoning_inconsistency']


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency
0,consolidation_normal_000,heldout_source_000,normal,normal,NORMAL,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5412,33.9204,"```json\n{\n ""participation_assessment"": ""VAL...",NaN,True,False,False,False,False,False
1,consolidation_normal_001,heldout_source_001,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,NORMAL,NORMAL,...,5274,33.6213,"```json\n{\n ""participation_assessment"": ""VAL...",NaN,True,False,False,False,False,False
2,consolidation_normal_002,heldout_source_002,normal,normal,NORMAL,ANOMALOUS,VALID,ANOMALOUS,ANOMALOUS,ANOMALOUS,...,5441,34.1377,"```json\n{\n ""participation_assessment"": ""VAL...",NaN,True,False,False,False,False,False
3,consolidation_normal_003,heldout_source_003,normal,normal,NORMAL,ANOMALOUS,VALID,NORMAL,ANOMALOUS,ANOMALOUS,...,5358,34.1833,"```json\n{\n ""participation_assessment"": ""VAL...",NaN,True,False,False,False,False,False
4,consolidation_normal_004,heldout_source_004,normal,normal,NORMAL,NORMAL,VALID,NORMAL,NORMAL,NORMAL,...,5479,33.8164,"```json\n{\n ""participation_assessment"": ""VAL...",NaN,True,True,False,False,False,False


In [ ]:
# ============================================================
# RESTORE AND EVALUATE R1 FROM THE FINAL SAVED CSV
# DOES NOT USE predictions_cache.json
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)


# ============================================================
# PATHS
# ============================================================

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/"
    "reasoning_r1_full_semantics_normal_references_only_chain_of_thought"
)


PREDICTIONS_CSV_PATH = (
    EXPERIMENT_DIR
    / "predictions_all_400.csv"
)


REASONING_ASSESSMENTS_CSV_PATH = (
    EXPERIMENT_DIR
    / "reasoning_assessments.csv"
)


ERRORS_CSV_PATH = (
    EXPERIMENT_DIR
    / "classification_errors.csv"
)


assert PREDICTIONS_CSV_PATH.exists(), (
    "Final predictions CSV was not found:\n"
    f"{PREDICTIONS_CSV_PATH}"
)


# ============================================================
# LOAD FINAL SAVED PREDICTIONS
# ============================================================

results_df = pd.read_csv(
    PREDICTIONS_CSV_PATH
)


assert len(results_df) == 400, (
    f"Expected 400 cases, but loaded {len(results_df)}."
)


assert results_df["case_id"].is_unique, (
    "Duplicate case IDs were found."
)


# ============================================================
# RESTORE BOOLEAN COLUMNS AFTER CSV LOADING
# ============================================================

def normalize_boolean_series(series):

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series
        .fillna(False)
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
        .astype(bool)
    )


if "valid_prediction" in results_df.columns:

    results_df[
        "valid_prediction"
    ] = normalize_boolean_series(
        results_df[
            "valid_prediction"
        ]
    )

else:

    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        [
            "NORMAL",
            "ANOMALOUS",
        ]
    )


if "schema_exact" in results_df.columns:

    results_df[
        "schema_exact"
    ] = normalize_boolean_series(
        results_df[
            "schema_exact"
        ]
    )

else:

    results_df[
        "schema_exact"
    ] = (
        results_df[
            "parse_mode"
        ]
        == "structured_json"
    )


if "correct" in results_df.columns:

    results_df[
        "correct"
    ] = normalize_boolean_series(
        results_df[
            "correct"
        ]
    )

else:

    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


if "reasoning_inconsistency" in results_df.columns:

    results_df[
        "reasoning_inconsistency"
    ] = normalize_boolean_series(
        results_df[
            "reasoning_inconsistency"
        ]
    )

else:

    results_df[
        "reasoning_inconsistency"
    ] = False


# ============================================================
# VALID AND INVALID SUBSETS
# ============================================================

valid_df = results_df[
    results_df[
        "valid_prediction"
    ]
].copy()


invalid_df = results_df[
    ~results_df[
        "valid_prediction"
    ]
].copy()


assert len(valid_df) > 0, (
    "No valid predictions were found."
)


y_true = valid_df[
    "gold_label"
].astype(str)


y_pred = valid_df[
    "prediction"
].astype(str)


LABEL_ORDER = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# CONFUSION MATRIX
# ============================================================

confusion_array = confusion_matrix(
    y_true,
    y_pred,
    labels=LABEL_ORDER,
)


confusion_df = pd.DataFrame(
    confusion_array,
    index=[
        "Gold NORMAL",
        "Gold ANOMALOUS",
    ],
    columns=[
        "Pred NORMAL",
        "Pred ANOMALOUS",
    ],
)


true_normal = int(
    confusion_array[0, 0]
)

false_anomalous = int(
    confusion_array[0, 1]
)

false_normal = int(
    confusion_array[1, 0]
)

true_anomalous = int(
    confusion_array[1, 1]
)


normal_recall = (
    true_normal
    /
    (
        true_normal
        +
        false_anomalous
    )
)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

classification_report_dict = classification_report(
    y_true,
    y_pred,
    labels=LABEL_ORDER,
    target_names=LABEL_ORDER,
    output_dict=True,
    zero_division=0,
)


classification_report_df = pd.DataFrame(
    classification_report_dict
).transpose()


# ============================================================
# MATCHED SOURCE-GROUP EXACT RATE
# ============================================================

if "source_group_id" in results_df.columns:

    source_group_exact_df = (
        results_df
        .groupby(
            "source_group_id",
            dropna=False,
        )[
            "correct"
        ]
        .all()
        .reset_index(
            name="all_cases_correct"
        )
    )


    source_group_exact_match_rate = float(
        source_group_exact_df[
            "all_cases_correct"
        ].mean()
    )

else:

    source_group_exact_df = pd.DataFrame()

    source_group_exact_match_rate = float(
        "nan"
    )


# ============================================================
# METRICS
# ============================================================

metrics = {
    "experiment_version": (
        R1_CONFIG.get(
            "experiment_version",
            "R1",
        )
        if "R1_CONFIG" in globals()
        else "R1"
    ),

    "total_cases": int(
        len(results_df)
    ),

    "valid_predictions": int(
        len(valid_df)
    ),

    "invalid_predictions": int(
        len(invalid_df)
    ),

    "exact_json_schema_rate": float(
        results_df[
            "schema_exact"
        ].mean()
    ),

    "reasoning_inconsistency_count": int(
        results_df[
            "reasoning_inconsistency"
        ].sum()
    ),

    "accuracy_valid_predictions": float(
        accuracy_score(
            y_true,
            y_pred,
        )
    ),

    "strict_accuracy_invalid_as_wrong": float(
        results_df[
            "correct"
        ].mean()
    ),

    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    ),

    "anomalous_precision": float(
        precision_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "anomalous_recall": float(
        recall_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "anomalous_f1": float(
        f1_score(
            y_true,
            y_pred,
            pos_label="ANOMALOUS",
            zero_division=0,
        )
    ),

    "normal_recall_specificity": float(
        normal_recall
    ),

    "matthews_correlation_coefficient": float(
        matthews_corrcoef(
            y_true,
            y_pred,
        )
    ),

    "source_group_exact_match_rate": float(
        source_group_exact_match_rate
    ),

    "confusion_matrix_label_order": (
        LABEL_ORDER
    ),

    "confusion_matrix": (
        confusion_array.tolist()
    ),
}


# ============================================================
# PER-GROUP METRICS FUNCTION
# ============================================================

def build_group_metrics(
    dataframe,
    group_column,
):

    rows = []


    for group_value, group_df in dataframe.groupby(
        group_column,
        dropna=False,
        sort=True,
    ):

        group_valid_df = group_df[
            group_df[
                "valid_prediction"
            ]
        ]


        total_cases = len(
            group_df
        )

        valid_predictions = len(
            group_valid_df
        )

        invalid_predictions = (
            total_cases
            -
            valid_predictions
        )


        predicted_normal = int(
            (
                group_valid_df[
                    "prediction"
                ]
                ==
                "NORMAL"
            ).sum()
        )


        predicted_anomalous = int(
            (
                group_valid_df[
                    "prediction"
                ]
                ==
                "ANOMALOUS"
            ).sum()
        )


        correct_predictions = int(
            group_df[
                "correct"
            ].sum()
        )


        if valid_predictions > 0:

            accuracy_on_valid = float(
                (
                    group_valid_df[
                        "gold_label"
                    ]
                    ==
                    group_valid_df[
                        "prediction"
                    ]
                ).mean()
            )

        else:

            accuracy_on_valid = float(
                "nan"
            )


        rows.append({
            group_column: group_value,

            "total_cases": int(
                total_cases
            ),

            "valid_predictions": int(
                valid_predictions
            ),

            "invalid_predictions": int(
                invalid_predictions
            ),

            "predicted_NORMAL": int(
                predicted_normal
            ),

            "predicted_ANOMALOUS": int(
                predicted_anomalous
            ),

            "correct_predictions": int(
                correct_predictions
            ),

            "accuracy_on_valid": (
                accuracy_on_valid
            ),

            "strict_accuracy_invalid_as_wrong": float(
                group_df[
                    "correct"
                ].mean()
            ),

            "exact_schema_rate": float(
                group_df[
                    "schema_exact"
                ].mean()
            ),
        })


    return pd.DataFrame(
        rows
    )


family_metrics_df = build_group_metrics(
    results_df,
    "case_family",
)


variant_metrics_df = build_group_metrics(
    results_df,
    "case_variant",
)


# ============================================================
# ASSESSMENT DISTRIBUTIONS
# ============================================================

ASSESSMENT_FIELDS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


assessment_distribution_rows = []


for assessment_field in ASSESSMENT_FIELDS:

    assert assessment_field in results_df.columns, (
        f"Missing assessment field: {assessment_field}"
    )


    value_counts = (
        results_df[
            assessment_field
        ]
        .fillna(
            "MISSING"
        )
        .value_counts(
            dropna=False
        )
    )


    for assessment_value, count in value_counts.items():

        assessment_distribution_rows.append({
            "assessment_field": (
                assessment_field
            ),

            "assessment_value": (
                assessment_value
            ),

            "count": int(
                count
            ),
        })


assessment_distributions_df = pd.DataFrame(
    assessment_distribution_rows
)


# ============================================================
# REASONING-ASSESSMENT AND ERROR DATAFRAMES
# ============================================================

reasoning_columns = [
    column
    for column in [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "chain_of_thought",
        "reasoning_inconsistency",
        "correct",
    ]
    if column in results_df.columns
]


reasoning_assessments_df = results_df[
    reasoning_columns
].copy()


errors_df = results_df[
    ~results_df[
        "correct"
    ]
].copy()


# ============================================================
# RESTORE COMPLETE R1_EVALUATION OBJECT
# ============================================================

R1_EVALUATION = {
    "results_df": results_df,
    "valid_df": valid_df,
    "invalid_df": invalid_df,
    "errors_df": errors_df,
    "metrics": metrics,
    "confusion_df": confusion_df,
    "classification_report_df": (
        classification_report_df
    ),
    "family_metrics_df": family_metrics_df,
    "variant_metrics_df": variant_metrics_df,
    "assessment_distributions_df": (
        assessment_distributions_df
    ),
    "reasoning_assessments_df": (
        reasoning_assessments_df
    ),
    "source_group_exact_df": (
        source_group_exact_df
    ),
}


# ============================================================
# DISPLAY THE SAME FULL EVALUATION REPORT
# ============================================================

print("=" * 88)

print(
    "R1 — Full Semantics, No LAG Profiles, "
    "Structured Reasoning + Chain of Thought — RESULTS"
)

print("=" * 88)


print(
    "Total cases:",
    metrics[
        "total_cases"
    ],
)


print(
    "Valid predictions:",
    metrics[
        "valid_predictions"
    ],
)


print(
    "Invalid predictions:",
    metrics[
        "invalid_predictions"
    ],
)


print(
    "Exact structured-schema rate:",
    f'{metrics["exact_json_schema_rate"]:.4f}',
)


print(
    "Accuracy:",
    f'{metrics["accuracy_valid_predictions"]:.4f}',
)


print(
    "Balanced accuracy:",
    f'{metrics["balanced_accuracy"]:.4f}',
)


print(
    "ANOMALOUS precision:",
    f'{metrics["anomalous_precision"]:.4f}',
)


print(
    "ANOMALOUS recall:",
    f'{metrics["anomalous_recall"]:.4f}',
)


print(
    "ANOMALOUS F1:",
    f'{metrics["anomalous_f1"]:.4f}',
)


print(
    "NORMAL recall / specificity:",
    f'{metrics["normal_recall_specificity"]:.4f}',
)


print(
    "MCC:",
    f'{metrics["matthews_correlation_coefficient"]:.4f}',
)


print(
    "Matched source-group exact rate:",
    f'{metrics["source_group_exact_match_rate"]:.4f}',
)


print(
    "Reasoning inconsistencies:",
    metrics[
        "reasoning_inconsistency_count"
    ],
)


print("\nCONFUSION MATRIX")

display(
    confusion_df
)


print("\nCLASSIFICATION REPORT")

display(
    classification_report_df
)


print("\nPER CASE FAMILY")

display(
    family_metrics_df
)


print("\nPER EXACT CASE VARIANT")

display(
    variant_metrics_df
)


print("\nASSESSMENT DISTRIBUTIONS")

display(
    assessment_distributions_df
)


print(
    "\nLoaded predictions:",
    PREDICTIONS_CSV_PATH,
)


print(
    "Original inference cache:",
    "DELETED — not required for this evaluation",
)

R1 — Full Semantics, No LAG Profiles, Structured Reasoning + Chain of Thought — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8675
Balanced accuracy: 0.7783
ANOMALOUS precision: 0.8777
ANOMALOUS recall: 0.9567
ANOMALOUS F1: 0.9155
NORMAL recall / specificity: 0.6000
MCC: 0.6241
Matched source-group exact rate: 0.5400
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,60,40
Gold ANOMALOUS,13,287



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.821918,0.600000,0.693642,100.0000
ANOMALOUS,0.877676,0.956667,0.915470,300.0000
accuracy,0.867500,0.867500,0.867500,0.8675
macro avg,0.849797,0.778333,0.804556,400.0000
weighted avg,0.863736,0.867500,0.860013,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,4,46,46,0.92,0.92,1.0
1,lag_3sec,50,50,0,2,48,48,0.96,0.96,1.0
2,normal,100,100,0,60,40,60,0.60,0.60,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,7,93,93,0.93,0.93,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,NORMAL,163
3,local_temporal_assessment,LIMITED,138
4,local_temporal_assessment,ANOMALOUS,99
5,global_temporal_assessment,ANOMALOUS,178
6,global_temporal_assessment,LIMITED,138
7,global_temporal_assessment,NORMAL,84
8,temporal_assessment,ANOMALOUS,178
9,temporal_assessment,LIMITED,138



Loaded predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/reasoning_r1_full_semantics_normal_references_only_chain_of_thought/predictions_all_400.csv
Original inference cache: DELETED — not required for this evaluation


In [ ]:
# ============================================================
# PREPARE PER-SUBSET STRUCTURED + FREE-REASONING ANALYSIS
#
# This version additionally restores and prints the EXACT
# model-facing local and global temporal features for every
# representative reasoning trace.
# ============================================================

import copy
import json
import re

import pandas as pd
from IPython.display import display


# ============================================================
# REQUIRED NOTEBOOK STATE
# ============================================================

assert "R1_EVALUATION" in globals(), (
    "R1_EVALUATION was not found. "
    "Run the restored evaluation cell first."
)

assert "consolidation_cases" in globals(), (
    "consolidation_cases was not found. "
    "Run the database-loading cells first."
)

assert "build_binary_model_input" in globals(), (
    "build_binary_model_input was not found. "
    "Run the model-input construction cells first."
)


# ============================================================
# LOAD THE EVALUATION DATAFRAME
# ============================================================

analysis_df = (
    R1_EVALUATION[
        "results_df"
    ]
    .copy()
)


assert len(
    analysis_df
) == 400, (
    f"Expected 400 evaluated cases, "
    f"found {len(analysis_df)}."
)


# ============================================================
# BUILD AN EXACT CASE-ID LOOKUP
# ============================================================

case_lookup = {
    str(
        case[
            "case_id"
        ]
    ): case

    for case
    in consolidation_cases
}


assert len(
    case_lookup
) == 400, (
    f"Expected 400 unique database cases, "
    f"found {len(case_lookup)}."
)


analysis_case_ids = set(
    analysis_df[
        "case_id"
    ]
    .astype(str)
)


database_case_ids = set(
    case_lookup.keys()
)


missing_database_cases = sorted(
    analysis_case_ids
    -
    database_case_ids
)


extra_database_cases = sorted(
    database_case_ids
    -
    analysis_case_ids
)


assert not missing_database_cases, (
    "Evaluation cases missing from consolidation_cases: "
    f"{missing_database_cases[:20]}"
)


assert not extra_database_cases, (
    "Database cases missing from the evaluation dataframe: "
    f"{extra_database_cases[:20]}"
)


# ============================================================
# REBUILD THE EXACT MODEL-FACING PAYLOAD FOR EVERY CASE
#
# This uses the same build_binary_model_input function that was
# used when constructing the Qwen prompt.
#
# In particular:
# - only the selected exact temporal fields are retained;
# - global best_event_coverage is converted from fraction to
#   best_event_coverage_percent;
# - the values therefore match the prompt representation.
# ============================================================

exact_payload_by_case_id = {}


for case_id in sorted(
    analysis_case_ids
):

    case = case_lookup[
        case_id
    ]

    payload = build_binary_model_input(
        case
    )


    assert isinstance(
        payload,
        dict,
    )


    assert (
        "local_temporal_features"
        in payload
    ), (
        "local_temporal_features missing from exact payload "
        f"for {case_id}"
    )


    assert (
        "global_shift_features"
        in payload
    ), (
        "global_shift_features missing from exact payload "
        f"for {case_id}"
    )


    assert isinstance(
        payload[
            "local_temporal_features"
        ],
        dict,
    )


    assert isinstance(
        payload[
            "global_shift_features"
        ],
        dict,
    )


    exact_payload_by_case_id[
        case_id
    ] = payload


assert len(
    exact_payload_by_case_id
) == 400


# ============================================================
# ATTACH THE EXACT MODEL-FACING TEMPORAL FEATURES
# ============================================================

analysis_df[
    "case_id"
] = (
    analysis_df[
        "case_id"
    ]
    .astype(str)
)


analysis_df[
    "local_temporal_features"
] = (
    analysis_df[
        "case_id"
    ]
    .map(
        lambda case_id: copy.deepcopy(
            exact_payload_by_case_id[
                case_id
            ][
                "local_temporal_features"
            ]
        )
    )
)


analysis_df[
    "global_shift_features"
] = (
    analysis_df[
        "case_id"
    ]
    .map(
        lambda case_id: copy.deepcopy(
            exact_payload_by_case_id[
                case_id
            ][
                "global_shift_features"
            ]
        )
    )
)


assert analysis_df[
    "local_temporal_features"
].apply(
    lambda value: isinstance(
        value,
        dict,
    )
).all()


assert analysis_df[
    "global_shift_features"
].apply(
    lambda value: isinstance(
        value,
        dict,
    )
).all()


# ============================================================
# REQUIRED EVALUATION COLUMNS
# ============================================================

REQUIRED_COLUMNS = [
    "case_id",
    "case_family",
    "case_variant",
    "gold_label",
    "prediction",

    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",

    "chain_of_thought",

    "local_temporal_features",
    "global_shift_features",
]


missing_columns = [
    column

    for column
    in REQUIRED_COLUMNS

    if column
    not in analysis_df.columns
]


assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)


# ============================================================
# NORMALISE IMPORTANT STRING COLUMNS
# ============================================================

UPPERCASE_COLUMNS = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


for column in UPPERCASE_COLUMNS:

    analysis_df[
        column
    ] = (
        analysis_df[
            column
        ]
        .fillna(
            "MISSING"
        )
        .astype(str)
        .str.strip()
        .str.upper()
    )


analysis_df[
    "case_family"
] = (
    analysis_df[
        "case_family"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


analysis_df[
    "case_variant"
] = (
    analysis_df[
        "case_variant"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


analysis_df[
    "chain_of_thought"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .fillna("")
    .astype(str)
)


# ============================================================
# REASONING-TRACE DIAGNOSTICS
# ============================================================

def count_words(
    text,
):

    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(
                text
            ),
        )
    )


def count_numbered_steps(
    text,
):

    return len(
        re.findall(
            r"(?:^|\s)[1-7][\.\)]",
            str(
                text
            ),
        )
    )


analysis_df[
    "reasoning_word_count"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .apply(
        count_words
    )
)


analysis_df[
    "reasoning_step_count"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .apply(
        count_numbered_steps
    )
)


analysis_df[
    "has_limited_assessment"
] = (
    analysis_df[
        [
            "local_temporal_assessment",
            "global_temporal_assessment",
            "temporal_assessment",
            "semantic_assessment",
        ]
    ]
    .eq(
        "LIMITED"
    )
    .any(
        axis=1
    )
)


analysis_df[
    "reasoning_mentions_limited"
] = (
    analysis_df[
        "chain_of_thought"
    ]
    .str.lower()
    .str.contains(
        r"\blimited\b",
        regex=True,
    )
)


analysis_df[
    "reasoning_mentions_final_label"
] = (
    analysis_df.apply(
        lambda row: (
            row[
                "prediction"
            ].lower()
            in
            row[
                "chain_of_thought"
            ].lower()
        ),
        axis=1,
    )
)


# ============================================================
# OPTIONAL TEMPORAL-TRACE DIAGNOSTICS
#
# These columns make it easier to compare what the trace says
# against the exact numerical temporal evidence.
# ============================================================

def safe_feature_value(
    feature_dict,
    key,
):

    if not isinstance(
        feature_dict,
        dict,
    ):
        return None

    return feature_dict.get(
        key
    )


analysis_df[
    "local_num_offsets"
] = (
    analysis_df[
        "local_temporal_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "num_signed_strict_offsets",
        )
    )
)


analysis_df[
    "local_offset_median_seconds"
] = (
    analysis_df[
        "local_temporal_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "offset_median_seconds",
        )
    )
)


analysis_df[
    "local_offset_p90_seconds"
] = (
    analysis_df[
        "local_temporal_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "offset_p90_seconds",
        )
    )
)


analysis_df[
    "local_percent_above_1_5"
] = (
    analysis_df[
        "local_temporal_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "percent_offsets_above_1_5_seconds",
        )
    )
)


analysis_df[
    "global_correction_shift_seconds"
] = (
    analysis_df[
        "global_shift_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "best_B_correction_shift_seconds",
        )
    )
)


analysis_df[
    "global_alignment_gain"
] = (
    analysis_df[
        "global_shift_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "alignment_score_gain_vs_zero",
        )
    )
)


analysis_df[
    "global_num_events"
] = (
    analysis_df[
        "global_shift_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "best_num_bilateral_events",
        )
    )
)


analysis_df[
    "global_event_coverage_percent"
] = (
    analysis_df[
        "global_shift_features"
    ]
    .apply(
        lambda features: safe_feature_value(
            features,
            "best_event_coverage_percent",
        )
    )
)


# ============================================================
# STRUCTURED FIELDS
# ============================================================

STRUCTURED_FIELDS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


SIGNATURE_FIELDS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
]


# ============================================================
# ASSESSMENT DISTRIBUTION
# ============================================================

def build_assessment_distribution(
    subset_df,
):

    rows = []


    for field in STRUCTURED_FIELDS:

        counts = (
            subset_df[
                field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            rows.append({
                "assessment_field": (
                    field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),

                "percentage": round(
                    100.0
                    * count
                    / len(
                        subset_df
                    ),
                    2,
                ),
            })


    return pd.DataFrame(
        rows
    )


# ============================================================
# SELECT REPRESENTATIVE REASONING TRACES
#
# First selects different complete structured signatures.
# If fewer than n signatures exist, fills the remaining slots
# deterministically with additional cases.
# ============================================================

def select_representative_cases(
    subset_df,
    n=10,
):

    if len(
        subset_df
    ) == 0:

        return subset_df.copy()


    work_df = subset_df.copy()


    work_df[
        "structured_signature"
    ] = (
        work_df[
            SIGNATURE_FIELDS
        ]
        .fillna(
            "MISSING"
        )
        .astype(str)
        .agg(
            " | ".join,
            axis=1,
        )
    )


    work_df = (
        work_df
        .sort_values(
            [
                "structured_signature",
                "case_variant",
                "case_id",
            ]
        )
    )


    selected_df = (
        work_df
        .drop_duplicates(
            subset=[
                "structured_signature"
            ],
            keep="first",
        )
        .head(
            n
        )
    )


    remaining_needed = (
        min(
            n,
            len(
                work_df
            ),
        )
        -
        len(
            selected_df
        )
    )


    if remaining_needed > 0:

        remaining_df = (
            work_df[
                ~work_df.index.isin(
                    selected_df.index
                )
            ]
            .head(
                remaining_needed
            )
        )


        selected_df = pd.concat(
            [
                selected_df,
                remaining_df,
            ],
            axis=0,
        )


    return selected_df.head(
        n
    )


# ============================================================
# PRINT REASONING TRACES
#
# Each trace now includes:
# - exact local temporal model input;
# - exact global temporal model input;
# - structured assessments;
# - free-form chain_of_thought;
# - word and step counts.
# ============================================================

def print_reasoning_traces(
    subset_df,
    n=10,
):

    selected_df = (
        select_representative_cases(
            subset_df,
            n=n,
        )
    )


    print(
        "\nREPRESENTATIVE MODEL-GENERATED REASONING TRACES:",
        len(
            selected_df
        ),
    )


    for trace_index, (_, row) in enumerate(
        selected_df.iterrows(),
        start=1,
    ):

        print(
            "\n"
            + "=" * 110
        )

        print(
            f"TRACE {trace_index}"
        )

        print(
            "=" * 110
        )


        print(
            "CASE ID:",
            row[
                "case_id"
            ],
        )


        print(
            "CASE FAMILY:",
            row[
                "case_family"
            ],
        )


        print(
            "CASE VARIANT:",
            row[
                "case_variant"
            ],
        )


        print(
            "GOLD LABEL:",
            row[
                "gold_label"
            ],
        )


        print(
            "PREDICTION:",
            row[
                "prediction"
            ],
        )


        # ----------------------------------------------------
        # Exact local temporal input
        # ----------------------------------------------------

        print(
            "\nLOCAL TEMPORAL FEATURES "
            "(EXACT MODEL INPUT)"
        )

        print(
            json.dumps(
                row[
                    "local_temporal_features"
                ],
                indent=2,
                ensure_ascii=False,
                allow_nan=False,
            )
        )


        # ----------------------------------------------------
        # Exact global temporal input
        # ----------------------------------------------------

        print(
            "\nGLOBAL SHIFT FEATURES "
            "(EXACT MODEL INPUT)"
        )

        print(
            json.dumps(
                row[
                    "global_shift_features"
                ],
                indent=2,
                ensure_ascii=False,
                allow_nan=False,
            )
        )


        # ----------------------------------------------------
        # Structured assessments
        # ----------------------------------------------------

        print(
            "\nSTRUCTURED ASSESSMENTS"
        )


        print(
            "Participation:",
            row[
                "participation_assessment"
            ],
        )


        print(
            "Local temporal:",
            row[
                "local_temporal_assessment"
            ],
        )


        print(
            "Global temporal:",
            row[
                "global_temporal_assessment"
            ],
        )


        print(
            "Combined temporal:",
            row[
                "temporal_assessment"
            ],
        )


        print(
            "Semantic:",
            row[
                "semantic_assessment"
            ],
        )


        print(
            "Decisive dimension:",
            row[
                "decisive_dimension"
            ],
        )


        # ----------------------------------------------------
        # Reasoning length
        # ----------------------------------------------------

        print(
            "\nREASONING LENGTH"
        )


        print(
            "Words:",
            row[
                "reasoning_word_count"
            ],
        )


        print(
            "Detected numbered steps:",
            row[
                "reasoning_step_count"
            ],
        )


        # ----------------------------------------------------
        # Free-form reasoning
        # ----------------------------------------------------

        print(
            "\nMODEL-GENERATED REASONING TRACE"
        )


        print(
            row[
                "chain_of_thought"
            ]
        )


# ============================================================
# COMPLETE ANALYSIS FOR ONE SUBSET
# ============================================================

def analyse_subset(
    subset_df,
    title,
    n_traces=10,
):

    subset_df = subset_df.copy()


    print(
        "\n"
        + "#" * 110
    )

    print(
        title
    )

    print(
        "#" * 110
    )


    print(
        "Number of cases:",
        len(
            subset_df
        ),
    )


    if len(
        subset_df
    ) == 0:

        print(
            "No cases found in this subset."
        )

        return


    # --------------------------------------------------------
    # Case variants
    # --------------------------------------------------------

    print(
        "\nCASE VARIANT DISTRIBUTION"
    )


    display(
        subset_df[
            "case_variant"
        ]
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "case_variant"
        )
        .reset_index(
            name="count"
        )
    )


    # --------------------------------------------------------
    # Individual structured fields
    # --------------------------------------------------------

    print(
        "\nSTRUCTURED-ASSESSMENT DISTRIBUTIONS"
    )


    display(
        build_assessment_distribution(
            subset_df
        )
    )


    # --------------------------------------------------------
    # Local x global
    # --------------------------------------------------------

    print(
        "\nLOCAL × GLOBAL TEMPORAL ASSESSMENT"
    )


    display(
        pd.crosstab(
            subset_df[
                "local_temporal_assessment"
            ],

            subset_df[
                "global_temporal_assessment"
            ],

            margins=True,
            dropna=False,
        )
    )


    # --------------------------------------------------------
    # Combined temporal x semantic
    # --------------------------------------------------------

    print(
        "\nCOMBINED TEMPORAL × SEMANTIC ASSESSMENT"
    )


    display(
        pd.crosstab(
            subset_df[
                "temporal_assessment"
            ],

            subset_df[
                "semantic_assessment"
            ],

            margins=True,
            dropna=False,
        )
    )


    # --------------------------------------------------------
    # Decisive dimension x temporal
    # --------------------------------------------------------

    print(
        "\nDECISIVE DIMENSION × TEMPORAL ASSESSMENT"
    )


    display(
        pd.crosstab(
            subset_df[
                "decisive_dimension"
            ],

            subset_df[
                "temporal_assessment"
            ],

            margins=True,
            dropna=False,
        )
    )


    # --------------------------------------------------------
    # Complete structured patterns
    # --------------------------------------------------------

    complete_patterns_df = (
        subset_df
        .groupby(
            SIGNATURE_FIELDS,
            dropna=False,
        )
        .size()
        .reset_index(
            name="count"
        )
        .sort_values(
            "count",
            ascending=False,
        )
    )


    complete_patterns_df[
        "percentage"
    ] = (
        100.0
        * complete_patterns_df[
            "count"
        ]
        / len(
            subset_df
        )
    ).round(
        2
    )


    print(
        "\nMOST FREQUENT COMPLETE STRUCTURED PATTERNS"
    )


    display(
        complete_patterns_df.head(
            20
        )
    )


    # --------------------------------------------------------
    # Exact temporal feature summary
    # --------------------------------------------------------

    print(
        "\nEXACT TEMPORAL-FEATURE SUMMARY"
    )


    temporal_summary_columns = [
        "local_num_offsets",
        "local_offset_median_seconds",
        "local_offset_p90_seconds",
        "local_percent_above_1_5",
        "global_correction_shift_seconds",
        "global_alignment_gain",
        "global_num_events",
        "global_event_coverage_percent",
    ]


    display(
        subset_df[
            temporal_summary_columns
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .describe()
        .round(
            4
        )
    )


    # --------------------------------------------------------
    # Temporal features by structured temporal assessment
    # --------------------------------------------------------

    print(
        "\nTEMPORAL FEATURES BY COMBINED TEMPORAL ASSESSMENT"
    )


    display(
        subset_df[
            [
                "temporal_assessment",
                *temporal_summary_columns,
            ]
        ]
        .groupby(
            "temporal_assessment",
            dropna=False,
        )
        .agg(
            {
                column: [
                    "count",
                    "mean",
                    "median",
                ]

                for column
                in temporal_summary_columns
            }
        )
        .round(
            4
        )
    )


    # --------------------------------------------------------
    # Reasoning-trace length
    # --------------------------------------------------------

    print(
        "\nREASONING-TRACE LENGTH STATISTICS"
    )


    display(
        subset_df[
            [
                "reasoning_word_count",
                "reasoning_step_count",
            ]
        ]
        .describe()
        .round(
            2
        )
    )


    # --------------------------------------------------------
    # LIMITED audit
    # --------------------------------------------------------

    limited_cases = int(
        subset_df[
            "has_limited_assessment"
        ].sum()
    )


    limited_mentions = int(
        (
            subset_df[
                "has_limited_assessment"
            ]
            &
            subset_df[
                "reasoning_mentions_limited"
            ]
        ).sum()
    )


    print(
        "\nLIMITED-EVIDENCE AUDIT"
    )


    print(
        "Cases containing at least one LIMITED assessment:",
        limited_cases,
    )


    print(
        "Of these, reasoning explicitly mentions LIMITED:",
        limited_mentions,
    )


    # --------------------------------------------------------
    # Final-label mention audit
    # --------------------------------------------------------

    print(
        "\nFINAL-LABEL MENTION AUDIT"
    )


    print(
        "Reasoning traces explicitly containing the predicted label:",
        int(
            subset_df[
                "reasoning_mentions_final_label"
            ].sum()
        ),
        "/",
        len(
            subset_df
        ),
    )


    # --------------------------------------------------------
    # Full representative traces with exact features
    # --------------------------------------------------------

    print_reasoning_traces(
        subset_df,
        n=n_traces,
    )


# ============================================================
# FINAL AUDIT
# ============================================================

print(
    "Prepared analysis dataframe:",
    analysis_df.shape
)


print(
    "Exact model-facing payloads restored:",
    len(
        exact_payload_by_case_id
    ),
)


print(
    "All cases have local temporal features:",
    bool(
        analysis_df[
            "local_temporal_features"
        ]
        .apply(
            lambda value: isinstance(
                value,
                dict,
            )
        )
        .all()
    ),
)


print(
    "All cases have global shift features:",
    bool(
        analysis_df[
            "global_shift_features"
        ]
        .apply(
            lambda value: isinstance(
                value,
                dict,
            )
        )
        .all()
    ),
)


print(
    "\nPredictions:"
)


display(
    analysis_df[
        "prediction"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "prediction"
    )
    .reset_index(
        name="count"
    )
)

Prepared analysis dataframe: (400, 41)
Exact model-facing payloads restored: 400
All cases have local temporal features: True
All cases have global shift features: True

Predictions:


,prediction,count
0,ANOMALOUS,327
1,NORMAL,73


In [ ]:
# ============================================================
# 1. CORRECTLY DETECTED LAG CASES
# GOLD ANOMALOUS — PREDICTED ANOMALOUS
# ============================================================

lag_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "lag"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    lag_correct_df,
    title=(
        "CORRECTLY DETECTED LAG CASES "
        "(Gold ANOMALOUS, Predicted ANOMALOUS)"
    ),
    n_traces=10,
)


##############################################################################################################
CORRECTLY DETECTED LAG CASES (Gold ANOMALOUS, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 94

CASE VARIANT DISTRIBUTION


,case_variant,count
0,lag_3sec,48
1,lag_2sec,46



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,94,100.00
1,local_temporal_assessment,ANOMALOUS,47,50.00
2,local_temporal_assessment,NORMAL,41,43.62
3,local_temporal_assessment,LIMITED,6,6.38
4,global_temporal_assessment,ANOMALOUS,88,93.62
5,global_temporal_assessment,LIMITED,6,6.38
6,temporal_assessment,ANOMALOUS,88,93.62
7,temporal_assessment,LIMITED,6,6.38
8,semantic_assessment,COMPATIBLE,94,100.00
9,decisive_dimension,TEMPORAL,94,100.00



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


TEMPORAL FEATURES BY COMBINED TEMPORAL ASSESSMENT


local_num_offsets                 \
                                count    mean median   
temporal_assessment                                    
ANOMALOUS                          88  4.8864    4.0   
LIMITED                             6  1.1667    1.0   

                    local_offset_median_seconds                 \
                                          count    mean median   
temporal_assessment                                              
ANOMALOUS                                    88  1.7442  1.735   
LIMITED                                       3 -0.1367 -0.040   

                    local_offset_p90_seconds                \
                                       count   mean median   
temporal_assessment                                          
ANOMALOUS                                 88  2.805  2.775   
LIMITED                                    3  1.370  1.370   

                    local_percent_above_1_5  ...  \
                                      count  ...   
temporal_assessment                          ...   
ANOMALOUS                                88  ...   
LIMITED                                   3  ...   

                    global_correction_shift_seconds global_alignment_gain  \
                                             median                 count   
temporal_assessment                                                         
ANOMALOUS                                      -2.5                    88   
LIMITED                                        -2.3                     6   

                                    global_num_events                  \
                       mean  median             count     mean median   
temporal_assessment                                                     
ANOMALOUS            0.1991  0.1979                88  11.2386   10.0   
LIMITED              0.2063  0.1962                 6   5.8333    5.5   

                    global_event_coverage_percent                    
                                            count     mean   median  
temporal_assessment                                                  
ANOMALOUS                                      88  56.5007  55.5556  
LIMITED                                         6  32.9678  34.8290  

[2 rows x 24 columns]


REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,94.00,94.00
mean,132.01,7.88
std,30.92,1.08
min,57.00,3.00
25%,113.00,7.00
50%,120.00,8.00
75%,157.75,9.00
max,218.00,10.00



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 6
Of these, reasoning explicitly mentions LIMITED: 6

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 94 / 94

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 10

TRACE 1
CASE ID: consolidation_lag_2sec_000
CASE FAMILY: lag
CASE VARIANT: lag_2sec
GOLD LABEL: ANOMALOUS
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 0.0,
  "clean_overlap_percent": 0.0,
  "signed_strict_offsets_seconds": [
    3.16,
    4.28
  ],
  "num_signed_strict_offsets": 2,
  "offset_mean_seconds": 3.72,
  "offset_median_seconds": 3.72,
  "offset_max_seconds": 4.28,
  "offset_p75_seconds": 4.0,
  "offset_p90_seconds": 4.17,
  "num_offsets_above_1_5_seconds": 2,
  "percent_offsets_above_1_5_seconds": 100.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 2.7,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs

In [ ]:
# ============================================================
# 2. MISSED LAG CASES
# GOLD ANOMALOUS — PREDICTED NORMAL
# ============================================================

lag_missed_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "lag"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


analyse_subset(
    lag_missed_df,
    title=(
        "MISSED LAG CASES "
        "(Gold ANOMALOUS, Predicted NORMAL)"
    ),
    n_traces=10,
)


##############################################################################################################
MISSED LAG CASES (Gold ANOMALOUS, Predicted NORMAL)
##############################################################################################################
Number of cases: 6

CASE VARIANT DISTRIBUTION


,case_variant,count
0,lag_2sec,4
1,lag_3sec,2



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,6,100.0
1,local_temporal_assessment,NORMAL,6,100.0
2,global_temporal_assessment,NORMAL,6,100.0
3,temporal_assessment,NORMAL,6,100.0
4,semantic_assessment,COMPATIBLE,6,100.0
5,decisive_dimension,SEMANTIC,6,100.0



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


TEMPORAL FEATURES BY COMBINED TEMPORAL ASSESSMENT


local_num_offsets             local_offset_median_seconds  \
                                count mean median                       count   
temporal_assessment                                                             
NORMAL                              6  4.5    4.0                           6   

                                 local_offset_p90_seconds                \
                     mean median                    count   mean median   
temporal_assessment                                                       
NORMAL               1.15    0.5                        6  2.175   1.46   

                    local_percent_above_1_5  ...  \
                                      count  ...   
temporal_assessment                          ...   
NORMAL                                    6  ...   

                    global_correction_shift_seconds global_alignment_gain  \
                                             median                 count   
temporal_assessment                                                         
NORMAL                                        -0.35                     6   

                                    global_num_events              \
                       mean  median             count mean median   
temporal_assessment                                                 
NORMAL               0.0752  0.0429                 6  8.5    8.0   

                    global_event_coverage_percent                    
                                            count     mean   median  
temporal_assessment                                                  
NORMAL                                          6  48.4452  46.0526  

[1 rows x 24 columns]


REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,6.00,6.0
mean,112.17,7.0
std,10.26,0.0
min,97.00,7.0
25%,106.25,7.0
50%,113.00,7.0
75%,119.00,7.0
max,125.00,7.0



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 0
Of these, reasoning explicitly mentions LIMITED: 0

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 6 / 6

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 6

TRACE 1
CASE ID: consolidation_lag_2sec_009
CASE FAMILY: lag
CASE VARIANT: lag_2sec
GOLD LABEL: ANOMALOUS
PREDICTION: NORMAL

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 5.12,
  "clean_overlap_percent": 4.27,
  "signed_strict_offsets_seconds": [
    4.25,
    0.09,
    0.47,
    2.46
  ],
  "num_signed_strict_offsets": 4,
  "offset_mean_seconds": 1.82,
  "offset_median_seconds": 1.46,
  "offset_max_seconds": 4.25,
  "offset_p75_seconds": 2.91,
  "offset_p90_seconds": 3.71,
  "num_offsets_above_1_5_seconds": 2,
  "percent_offsets_above_1_5_seconds": 50.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": -0.3,
  "estimated_B_lateness_seconds": 0.3,
  "alignm

In [ ]:
# ============================================================
# 3. CORRECTLY DETECTED WRONG-PARTNER CASES
# GOLD ANOMALOUS — PREDICTED ANOMALOUS
# ============================================================

wrong_partner_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "wrong_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    wrong_partner_correct_df,
    title=(
        "CORRECTLY DETECTED WRONG-PARTNER CASES "
        "(Gold ANOMALOUS, Predicted ANOMALOUS)"
    ),
    n_traces=10,
)


##############################################################################################################
CORRECTLY DETECTED WRONG-PARTNER CASES (Gold ANOMALOUS, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 93

CASE VARIANT DISTRIBUTION


,case_variant,count
0,wrong_partner,93



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,93,100.00
1,local_temporal_assessment,ANOMALOUS,38,40.86
2,local_temporal_assessment,NORMAL,29,31.18
3,local_temporal_assessment,LIMITED,26,27.96
4,global_temporal_assessment,ANOMALOUS,64,68.82
5,global_temporal_assessment,LIMITED,26,27.96
6,global_temporal_assessment,NORMAL,3,3.23
7,temporal_assessment,ANOMALOUS,64,68.82
8,temporal_assessment,LIMITED,26,27.96
9,temporal_assessment,NORMAL,3,3.23



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


TEMPORAL FEATURES BY COMBINED TEMPORAL ASSESSMENT


local_num_offsets                 \
                                count    mean median   
temporal_assessment                                    
ANOMALOUS                          64  3.4844    3.0   
LIMITED                            26  1.7692    2.0   
NORMAL                              3  5.3333    6.0   

                    local_offset_median_seconds                 \
                                          count    mean median   
temporal_assessment                                              
ANOMALOUS                                    64  1.0920  0.735   
LIMITED                                      24  0.1925  0.185   
NORMAL                                        3  0.8867  0.950   

                    local_offset_p90_seconds                 \
                                       count    mean median   
temporal_assessment                                           
ANOMALOUS                                 64  2.3552  2.355   
LIMITED                                   24  0.7012  0.645   
NORMAL                                     3  2.8367  2.830   

                    local_percent_above_1_5  ...  \
                                      count  ...   
temporal_assessment                          ...   
ANOMALOUS                                64  ...   
LIMITED                                  24  ...   
NORMAL                                    3  ...   

                    global_correction_shift_seconds global_alignment_gain  \
                                             median                 count   
temporal_assessment                                                         
ANOMALOUS                                      0.25                    64   
LIMITED                                        0.05                    26   
NORMAL                                        -0.40                     3   

                                    global_num_events                  \
                       mean  median             count     mean median   
temporal_assessment                                                     
ANOMALOUS            0.0903  0.0773                64   8.7969    8.0   
LIMITED              0.0874  0.0779                26   6.3462    6.0   
NORMAL               0.0236  0.0260                 3  11.3333   12.0   

                    global_event_coverage_percent                    
                                            count     mean   median  
temporal_assessment                                                  
ANOMALOUS                                      64  40.9703  42.2065  
LIMITED                                        26  34.5499  33.3333  
NORMAL                                          3  51.8446  51.7241  

[3 rows x 24 columns]


REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,93.00,93.00
mean,135.84,8.03
std,27.62,1.02
min,76.00,7.00
25%,117.00,7.00
50%,135.00,8.00
75%,157.00,9.00
max,202.00,11.00



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 32
Of these, reasoning explicitly mentions LIMITED: 32

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 92 / 93

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 10

TRACE 1
CASE ID: consolidation_wrong_partner_003
CASE FAMILY: wrong_partner
CASE VARIANT: wrong_partner
GOLD LABEL: ANOMALOUS
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 45.82,
  "clean_overlap_percent": 38.18,
  "signed_strict_offsets_seconds": [
    2.69
  ],
  "num_signed_strict_offsets": 1,
  "offset_mean_seconds": 2.69,
  "offset_median_seconds": 2.69,
  "offset_max_seconds": 2.69,
  "offset_p75_seconds": 2.69,
  "offset_p90_seconds": 2.69,
  "num_offsets_above_1_5_seconds": 1,
  "percent_offsets_above_1_5_seconds": 100.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": -2.7,
  "estimated_B_lateness_seconds": 2.7,
  "align

In [ ]:
# ============================================================
# 4. MISSED WRONG-PARTNER CASES
# GOLD ANOMALOUS — PREDICTED NORMAL
# ============================================================

wrong_partner_missed_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "wrong_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


analyse_subset(
    wrong_partner_missed_df,
    title=(
        "MISSED WRONG-PARTNER CASES "
        "(Gold ANOMALOUS, Predicted NORMAL)"
    ),
    n_traces=10,
)


##############################################################################################################
MISSED WRONG-PARTNER CASES (Gold ANOMALOUS, Predicted NORMAL)
##############################################################################################################
Number of cases: 7

CASE VARIANT DISTRIBUTION


,case_variant,count
0,wrong_partner,7



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,7,100.0
1,local_temporal_assessment,NORMAL,7,100.0
2,global_temporal_assessment,NORMAL,7,100.0
3,temporal_assessment,NORMAL,7,100.0
4,semantic_assessment,COMPATIBLE,7,100.0
5,decisive_dimension,SEMANTIC,7,100.0



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


MOST FREQUENT COMPLETE STRUCTURED PATTERNS


,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,count,percentage
0,VALID,NORMAL,NORMAL,NORMAL,COMPATIBLE,SEMANTIC,7,100.0



REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,7.00,7.0
mean,116.86,7.0
std,8.76,0.0
min,105.00,7.0
25%,110.00,7.0
50%,121.00,7.0
75%,121.50,7.0
max,129.00,7.0



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 0
Of these, reasoning explicitly mentions LIMITED: 0

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 7 / 7

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 7

TRACE 1
CASE ID: consolidation_wrong_partner_006
CASE FAMILY: wrong_partner
CASE VARIANT: wrong_partner
GOLD LABEL: ANOMALOUS
PREDICTION: NORMAL

STRUCTURED ASSESSMENTS
Participation: VALID
Local temporal: NORMAL
Global temporal: NORMAL
Combined temporal: NORMAL
Semantic: COMPATIBLE
Decisive dimension: SEMANTIC

REASONING LENGTH
Words: 105
Detected numbered steps: 7

MODEL-GENERATED REASONING TRACE
1. Both participants speak during the 120-second interval, meeting participation validity. 
2. The local temporal evidence shows a signed-offset distribution consistent with the frozen NORMAL pattern, with no repeated delayed handoffs. 
3. The global temporal evidence shows a preferred correction near zero, with limited est

In [ ]:
# ============================================================
# 5. CORRECTLY DETECTED NORMAL CASES
# GOLD NORMAL — PREDICTED NORMAL
# ============================================================

normal_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "normal"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


analyse_subset(
    normal_correct_df,
    title=(
        "CORRECTLY DETECTED NORMAL CASES "
        "(Gold NORMAL, Predicted NORMAL)"
    ),
    n_traces=10,
)


##############################################################################################################
CORRECTLY DETECTED NORMAL CASES (Gold NORMAL, Predicted NORMAL)
##############################################################################################################
Number of cases: 60

CASE VARIANT DISTRIBUTION


,case_variant,count
0,normal,60



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,60,100.0
1,local_temporal_assessment,NORMAL,60,100.0
2,global_temporal_assessment,NORMAL,60,100.0
3,temporal_assessment,NORMAL,60,100.0
4,semantic_assessment,COMPATIBLE,60,100.0
5,decisive_dimension,SEMANTIC,60,100.0



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


MOST FREQUENT COMPLETE STRUCTURED PATTERNS


,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,count,percentage
0,VALID,NORMAL,NORMAL,NORMAL,COMPATIBLE,SEMANTIC,60,100.0



REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,60.00,60.00
mean,120.62,7.03
std,11.20,0.18
min,95.00,7.00
25%,111.75,7.00
50%,121.50,7.00
75%,129.25,7.00
max,146.00,8.00



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 0
Of these, reasoning explicitly mentions LIMITED: 0

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 60 / 60

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 10

TRACE 1
CASE ID: consolidation_normal_004
CASE FAMILY: normal
CASE VARIANT: normal
GOLD LABEL: NORMAL
PREDICTION: NORMAL

STRUCTURED ASSESSMENTS
Participation: VALID
Local temporal: NORMAL
Global temporal: NORMAL
Combined temporal: NORMAL
Semantic: COMPATIBLE
Decisive dimension: SEMANTIC

REASONING LENGTH
Words: 117
Detected numbered steps: 7

MODEL-GENERATED REASONING TRACE
1. Both participants speak during the 120-second interval, meeting participation validity. 
2. The local temporal evidence shows a signed-offset distribution consistent with the frozen NORMAL pattern, with no repeated delayed handoffs. 
3. The global temporal evidence shows a preferred correction near zero, limited estimated lateness, and limit

In [ ]:
# ============================================================
# 6. FALSE-POSITIVE NORMAL CASES
# GOLD NORMAL — PREDICTED ANOMALOUS
# ============================================================

normal_false_positive_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "normal"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    normal_false_positive_df,
    title=(
        "FALSE-POSITIVE NORMAL CASES "
        "(Gold NORMAL, Predicted ANOMALOUS)"
    ),
    n_traces=10,
)


##############################################################################################################
FALSE-POSITIVE NORMAL CASES (Gold NORMAL, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 40

CASE VARIANT DISTRIBUTION


,case_variant,count
0,normal,40



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,VALID,40,100.0
1,local_temporal_assessment,NORMAL,20,50.0
2,local_temporal_assessment,ANOMALOUS,14,35.0
3,local_temporal_assessment,LIMITED,6,15.0
4,global_temporal_assessment,ANOMALOUS,26,65.0
5,global_temporal_assessment,NORMAL,8,20.0
6,global_temporal_assessment,LIMITED,6,15.0
7,temporal_assessment,ANOMALOUS,26,65.0
8,temporal_assessment,NORMAL,8,20.0
9,temporal_assessment,LIMITED,6,15.0



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


TEMPORAL FEATURES BY COMBINED TEMPORAL ASSESSMENT


local_num_offsets                 \
                                count    mean median   
temporal_assessment                                    
ANOMALOUS                          26  4.3846    4.5   
LIMITED                             6  1.8333    2.0   
NORMAL                              8  7.0000    7.0   

                    local_offset_median_seconds                 \
                                          count    mean median   
temporal_assessment                                              
ANOMALOUS                                    26  0.8035   0.74   
LIMITED                                       5 -0.0480   0.15   
NORMAL                                        8  0.3225   0.48   

                    local_offset_p90_seconds                 \
                                       count    mean median   
temporal_assessment                                           
ANOMALOUS                                 26  1.6215  1.810   
LIMITED                                    5  0.2260  0.660   
NORMAL                                     8  1.1388  1.385   

                    local_percent_above_1_5  ...  \
                                      count  ...   
temporal_assessment                          ...   
ANOMALOUS                                26  ...   
LIMITED                                   5  ...   
NORMAL                                    8  ...   

                    global_correction_shift_seconds global_alignment_gain  \
                                             median                 count   
temporal_assessment                                                         
ANOMALOUS                                       0.7                    26   
LIMITED                                         0.2                     6   
NORMAL                                         -0.0                     8   

                                    global_num_events                  \
                       mean  median             count     mean median   
temporal_assessment                                                     
ANOMALOUS            0.0608  0.0565                26   9.5385   10.0   
LIMITED              0.0740  0.0310                 6   4.6667    5.0   
NORMAL               0.0093  0.0034                 8  13.7500   13.5   

                    global_event_coverage_percent                    
                                            count     mean   median  
temporal_assessment                                                  
ANOMALOUS                                      26  50.3895  50.1253  
LIMITED                                         6  38.9156  40.1960  
NORMAL                                          8  62.9501  62.1978  

[3 rows x 24 columns]


REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,40.00,40.00
mean,132.93,7.92
std,28.10,1.14
min,86.00,7.00
25%,117.25,7.00
50%,129.50,7.50
75%,139.50,9.00
max,239.00,11.00



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 7
Of these, reasoning explicitly mentions LIMITED: 7

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 40 / 40

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 10

TRACE 1
CASE ID: consolidation_normal_000
CASE FAMILY: normal
CASE VARIANT: normal
GOLD LABEL: NORMAL
PREDICTION: ANOMALOUS

LOCAL TEMPORAL FEATURES (EXACT MODEL INPUT)
{
  "clean_overlap_seconds": 0.0,
  "clean_overlap_percent": 0.0,
  "signed_strict_offsets_seconds": [
    1.16,
    2.28
  ],
  "num_signed_strict_offsets": 2,
  "offset_mean_seconds": 1.72,
  "offset_median_seconds": 1.72,
  "offset_max_seconds": 2.28,
  "offset_p75_seconds": 2.0,
  "offset_p90_seconds": 2.17,
  "num_offsets_above_1_5_seconds": 1,
  "percent_offsets_above_1_5_seconds": 50.0
}

GLOBAL SHIFT FEATURES (EXACT MODEL INPUT)
{
  "best_B_correction_shift_seconds": 4.7,
  "estimated_B_lateness_seconds": 0.0,
  "alignment_score_gain_vs_zero

In [ ]:
# ============================================================
# 7. CORRECTLY DETECTED SILENT-PARTNER CASES
# GOLD ANOMALOUS — PREDICTED ANOMALOUS
# ============================================================

silent_correct_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "silent_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "ANOMALOUS"
    )
].copy()


analyse_subset(
    silent_correct_df,
    title=(
        "CORRECTLY DETECTED SILENT-PARTNER CASES "
        "(Gold ANOMALOUS, Predicted ANOMALOUS)"
    ),
    n_traces=5,
)


silent_missed_df = analysis_df[
    (
        analysis_df[
            "case_family"
        ]
        == "silent_partner"
    )
    &
    (
        analysis_df[
            "prediction"
        ]
        == "NORMAL"
    )
].copy()


print(
    "\nMissed silent-partner cases:",
    len(
        silent_missed_df
    ),
)


##############################################################################################################
CORRECTLY DETECTED SILENT-PARTNER CASES (Gold ANOMALOUS, Predicted ANOMALOUS)
##############################################################################################################
Number of cases: 100

CASE VARIANT DISTRIBUTION


,case_variant,count
0,silent_partner,100



STRUCTURED-ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count,percentage
0,participation_assessment,INVALID,100,100.0
1,local_temporal_assessment,LIMITED,100,100.0
2,global_temporal_assessment,LIMITED,100,100.0
3,temporal_assessment,LIMITED,100,100.0
4,semantic_assessment,LIMITED,94,94.0
5,semantic_assessment,COMPATIBLE,6,6.0
6,decisive_dimension,PARTICIPATION,100,100.0



LOCAL × GLOBAL TEMPORAL ASSESSMENT


...


MOST FREQUENT COMPLETE STRUCTURED PATTERNS


,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,semantic_assessment,decisive_dimension,count,percentage
1,INVALID,LIMITED,LIMITED,LIMITED,LIMITED,PARTICIPATION,94,94.0
0,INVALID,LIMITED,LIMITED,LIMITED,COMPATIBLE,PARTICIPATION,6,6.0



REASONING-TRACE LENGTH STATISTICS


,reasoning_word_count,reasoning_step_count
count,100.00,100.00
mean,119.13,6.53
std,22.74,0.59
min,71.00,5.00
25%,104.00,6.00
50%,110.50,7.00
75%,139.50,7.00
max,164.00,7.00



LIMITED-EVIDENCE AUDIT
Cases containing at least one LIMITED assessment: 100
Of these, reasoning explicitly mentions LIMITED: 100

FINAL-LABEL MENTION AUDIT
Reasoning traces explicitly containing the predicted label: 100 / 100

REPRESENTATIVE MODEL-GENERATED REASONING TRACES: 5

TRACE 1
CASE ID: consolidation_silent_partner_004
CASE FAMILY: silent_partner
CASE VARIANT: silent_partner
GOLD LABEL: ANOMALOUS
PREDICTION: ANOMALOUS

STRUCTURED ASSESSMENTS
Participation: INVALID
Local temporal: LIMITED
Global temporal: LIMITED
Combined temporal: LIMITED
Semantic: COMPATIBLE
Decisive dimension: PARTICIPATION

REASONING LENGTH
Words: 138
Detected numbered steps: 7

MODEL-GENERATED REASONING TRACE
1. Participation: Participant A speaks, but Participant B does not. This violates the requirement for both participants to contribute speech during the complete interval. 2. Local Temporal Evidence: The signed strict offsets list is empty, indicating no valid offsets. This suggests a lack of coordina

In [ ]:
# ============================================================
# 8. SUMMARY ACROSS ALL SUBSETS
# ============================================================

analysis_groups = {
    "lag_correct": lag_correct_df,
    "lag_missed": lag_missed_df,
    "wrong_partner_correct": wrong_partner_correct_df,
    "wrong_partner_missed": wrong_partner_missed_df,
    "normal_correct": normal_correct_df,
    "normal_false_positive": normal_false_positive_df,
    "silent_partner_correct": silent_correct_df,
}


summary_rows = []


for group_name, group_df in analysis_groups.items():

    summary_rows.append({
        "analysis_group": group_name,

        "num_cases": len(
            group_df
        ),

        "participation_INVALID": int(
            (
                group_df[
                    "participation_assessment"
                ]
                == "INVALID"
            ).sum()
        ),

        "local_NORMAL": int(
            (
                group_df[
                    "local_temporal_assessment"
                ]
                == "NORMAL"
            ).sum()
        ),

        "local_ANOMALOUS": int(
            (
                group_df[
                    "local_temporal_assessment"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "local_LIMITED": int(
            (
                group_df[
                    "local_temporal_assessment"
                ]
                == "LIMITED"
            ).sum()
        ),

        "global_NORMAL": int(
            (
                group_df[
                    "global_temporal_assessment"
                ]
                == "NORMAL"
            ).sum()
        ),

        "global_ANOMALOUS": int(
            (
                group_df[
                    "global_temporal_assessment"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "global_LIMITED": int(
            (
                group_df[
                    "global_temporal_assessment"
                ]
                == "LIMITED"
            ).sum()
        ),

        "temporal_NORMAL": int(
            (
                group_df[
                    "temporal_assessment"
                ]
                == "NORMAL"
            ).sum()
        ),

        "temporal_ANOMALOUS": int(
            (
                group_df[
                    "temporal_assessment"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "temporal_LIMITED": int(
            (
                group_df[
                    "temporal_assessment"
                ]
                == "LIMITED"
            ).sum()
        ),

        "semantic_COMPATIBLE": int(
            (
                group_df[
                    "semantic_assessment"
                ]
                == "COMPATIBLE"
            ).sum()
        ),

        "semantic_INCOMPATIBLE": int(
            (
                group_df[
                    "semantic_assessment"
                ]
                == "INCOMPATIBLE"
            ).sum()
        ),

        "semantic_LIMITED": int(
            (
                group_df[
                    "semantic_assessment"
                ]
                == "LIMITED"
            ).sum()
        ),

        "decisive_PARTICIPATION": int(
            (
                group_df[
                    "decisive_dimension"
                ]
                == "PARTICIPATION"
            ).sum()
        ),

        "decisive_TEMPORAL": int(
            (
                group_df[
                    "decisive_dimension"
                ]
                == "TEMPORAL"
            ).sum()
        ),

        "decisive_SEMANTIC": int(
            (
                group_df[
                    "decisive_dimension"
                ]
                == "SEMANTIC"
            ).sum()
        ),

        "mean_reasoning_words": round(
            group_df[
                "reasoning_word_count"
            ].mean(),
            2,
        ),
    })


subset_summary_df = pd.DataFrame(
    summary_rows
)


display(
    subset_summary_df
)

,analysis_group,num_cases,participation_INVALID,local_NORMAL,local_ANOMALOUS,local_LIMITED,global_NORMAL,global_ANOMALOUS,global_LIMITED,temporal_NORMAL,temporal_ANOMALOUS,temporal_LIMITED,semantic_COMPATIBLE,semantic_INCOMPATIBLE,semantic_LIMITED,decisive_PARTICIPATION,decisive_TEMPORAL,decisive_SEMANTIC,mean_reasoning_words
0,lag_correct,94,0,41,47,6,0,88,6,0,88,6,94,0,0,0,94,0,132.01
1,lag_missed,6,0,6,0,0,6,0,0,6,0,0,6,0,0,0,0,6,112.17
2,wrong_partner_correct,93,0,29,38,26,3,64,26,3,64,26,63,9,21,0,75,18,135.84
3,wrong_partner_missed,7,0,7,0,0,7,0,0,7,0,0,7,0,0,0,0,7,116.86
4,normal_correct,60,0,60,0,0,60,0,0,60,0,0,60,0,0,0,0,60,120.62
5,normal_false_positive,40,0,20,14,6,8,26,6,8,26,6,39,0,1,0,32,8,132.93
6,silent_partner_correct,100,100,0,0,100,0,0,100,0,0,100,6,0,94,100,0,0,119.13
